# RL 训练示例

演示使用 PPO 算法训练智能体控制车辆：
1. 环境与 PPO 配置
2. 训练过程
3. 训练曲线可视化
4. 加载模型并评估
5. 评估轨迹可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path

from hydra import compose, initialize_config_dir
from sim_env import RoadVehicleEnv
from rl import PPOAgent

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

## 1. 环境与 PPO 配置

创建训练环境和 PPO 超参数。这里使用较小的 `total_timesteps` 方便演示，实际训练可适当增大。

In [ ]:
conf_dir = (Path.cwd() / "../src/rl/conf").resolve()
if conf_dir is None:
    raise FileNotFoundError("未找到 Hydra 配置目录 src/rl/conf")

with initialize_config_dir(version_base=None, config_dir=str(conf_dir)):
    rl_cfg = compose(config_name="config")

log_dir = (Path.cwd() / "../log").resolve()
rl_cfg.log.save_dir = f"{log_dir}/rl_tutorial_{time.strftime('%Y%m%d_%H%M%S')}"
rl_cfg.agent.total_timesteps = 10000

## 3. 手动训练循环（可选）

如果需要更细粒度的控制（如记录训练曲线），可以手动执行采集-优化循环。

In [ ]:
env = RoadVehicleEnv.bulid_from_config(rl_cfg.env)
agent = PPOAgent(env, rl_cfg.agent)

reward_history = []
pg_loss_history = []
vf_loss_history = []
entropy_history = []

total_steps = 0
iteration = 0

while total_steps < rl_cfg.agent.total_timesteps:
    iteration += 1

    # 采集
    rollout_stats = agent.collect_rollout()
    total_steps += rl_cfg.agent.steps_per_epoch

    # 优化
    update_stats = agent.update()

    # 记录
    reward_history.append(rollout_stats["mean_reward"])
    pg_loss_history.append(update_stats["pg_loss"])
    vf_loss_history.append(update_stats["vf_loss"])
    entropy_history.append(update_stats["entropy"])

    if iteration % rl_cfg.log.log_interval == 0:
        print(
            f"[iter {iteration:4d}] "
            f"steps={total_steps:>8d}  "
            f"ep_reward={rollout_stats['mean_reward']:7.1f}  "
            f"episodes={rollout_stats['num_episodes']:3d}  "
            f"pg_loss={update_stats['pg_loss']:.4f}  "
            f"vf_loss={update_stats['vf_loss']:.4f}  "
            f"entropy={update_stats['entropy']:.4f}"
        )

print(f"\n训练完成，共 {iteration} 轮，{total_steps} 步")

## 4. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(reward_history, color="steelblue")
axes[0, 0].set_title("平均 Episode 奖励")
axes[0, 0].set_xlabel("迭代")
axes[0, 0].set_ylabel("奖励")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(pg_loss_history, color="coral")
axes[0, 1].set_title("策略损失 (Policy Loss)")
axes[0, 1].set_xlabel("迭代")
axes[0, 1].set_ylabel("损失")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(vf_loss_history, color="mediumseagreen")
axes[1, 0].set_title("价值损失 (Value Loss)")
axes[1, 0].set_xlabel("迭代")
axes[1, 0].set_ylabel("损失")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(entropy_history, color="orchid")
axes[1, 1].set_title("策略熵 (Entropy)")
axes[1, 1].set_xlabel("迭代")
axes[1, 1].set_ylabel("熵")
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle("PPO 训练曲线", fontsize=14)
plt.tight_layout()
plt.show()

## 5. 模型保存与加载

In [ ]:
# 保存
save_path = f"{rl_cfg.log.save_dir}/rl_toturial.pt"
agent.save(save_path)
print(f"模型已保存到 {save_path}")

# 加载
env_loaded = RoadVehicleEnv.bulid_from_config(rl_cfg.env)
agent_loaded = PPOAgent(env, rl_cfg.agent)

# save_path = "../log/rl_20260424_221506/ppo_983040.pt"
agent_loaded.load(save_path)
print("模型加载成功")

## 6. 评估与轨迹可视化

使用训练好的模型运行若干 episode，统计奖励并可视化轨迹。

In [ ]:
num_eval_episodes = 5
eval_rewards = []
eval_trajectories = []  # 存储每个 episode 的 (xs, ys)

for ep in range(num_eval_episodes):
    obs, info = env_loaded.reset(seed=ep)
    ep_reward = 0.0
    xs, ys = [info["x"]], [info["y"]]

    for _ in range(env_loaded.config.max_steps):
        action = agent_loaded.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env_loaded.step(action)
        ep_reward += reward
        xs.append(info["x"])
        ys.append(info["y"])
        if terminated or truncated:
            break

    eval_rewards.append(ep_reward)
    eval_trajectories.append((xs, ys))
    print(f"Episode {ep}: reward={ep_reward:.1f}, steps={len(xs)-1}, progress={info.get('progress', 0):.1%}")

env_loaded.close()
print(f"\n平均奖励: {np.mean(eval_rewards):.1f} ± {np.std(eval_rewards):.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.tab10(np.linspace(0, 1, num_eval_episodes))
for i, (xs, ys) in enumerate(eval_trajectories):
    ax.plot(xs, ys, color=colors[i], linewidth=1.5, label=f"Episode {i} (R={eval_rewards[i]:.0f})")
    ax.plot(xs[0], ys[0], "o", color=colors[i], markersize=8)
    ax.plot(xs[-1], ys[-1], "s", color=colors[i], markersize=8)

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("评估轨迹 (○=起点, □=终点)")
ax.legend(loc="best", fontsize=9)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()